# AVIRIS-3 L1B radiance download

Download an example AVIRIS-3 L1B calibrated radiance granule (`AV3_L1B_RDN_2356`) over the Permian Basin using NASA `earthaccess`. Each granule contains the radiance (`RDN`), observation geometry (`OBS`) and band mask NetCDF files plus a quicklook and metadata; only `RDN` and `OBS` are downloaded.

In [1]:
from pathlib import Path

import earthaccess
import xarray as xr

DATA_DIR = Path("aviris3")

# Permian Basin, GHG campaign, 2023-10-06
GRANULE = "AV320231006t181013_005_L1B_RDN_1"
SUFFIXES = ("_RDN.nc", "_OBS.nc")

## Login

Uses Earthdata credentials from environment variables, `~/.netrc`, or an interactive prompt. Pass `persist=True` to save them to `~/.netrc`.

In [2]:
auth = earthaccess.login(strategy="all")
assert auth.authenticated

## Search

In [4]:
results = earthaccess.search_data(version="1", granule_name=GRANULE)
assert len(results) == 1, f"found {len(results)} granules"
granule = results[0]
for url in granule.data_links():
    print(url.rsplit("/", 1)[-1])

AV320231006t181013_005_L1B_RDN_cbeae6f8_RDN.nc
AV320231006t181013_005_L1B_ORT_3b50f254_OBS.nc
AV320231006t181013_005_L1B_RDN_cbeae6f8_BANDMASK.nc
AV320231006t181013_005_L1B_RDN_cbeae6f8_RDN_QL.tif
AV320231006t181013_005_L1B_RDN_cbeae6f8.yaml
AV320231006t181013_005_L1B_ORT_3b50f254.yaml


## Download

Only the `RDN` and `OBS` files are kept; pass `granule` instead of `urls` to download all files.

In [5]:
urls = [url for url in granule.data_links() if url.endswith(SUFFIXES)]
files = [Path(p) for p in earthaccess.download(urls, local_path=DATA_DIR)]

files

QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2 [00:00<?, ?it/s]

[PosixPath('aviris3/AV320231006t181013_005_L1B_RDN_cbeae6f8_RDN.nc'),
 PosixPath('aviris3/AV320231006t181013_005_L1B_ORT_3b50f254_OBS.nc')]

## Quick check

In [6]:
rad_path = next(p for p in files if p.name.endswith("_RDN.nc"))
xr.open_datatree(rad_path)

<xarray.DataTree>
Group: /
│   Dimensions:              (lines: 1280, samples: 1234)
│   Dimensions without coordinates: lines, samples
│   Data variables:
│       lat                  (lines, samples) float32 6MB ...
│       lon                  (lines, samples) float32 6MB ...
│       elev                 (lines, samples) float32 6MB ...
│       transverse_mercator  |S1 1B ...
│   Attributes: (12/23)
│       Conventions:                       CF-1.6
│       date_created:                      2024-10-01T00:15:29Z
│       summary:                           The Airborne Visible / Infrared Imagin...
│       keywords:                          Imaging Spectroscopy, AVIRIS, AVIRIS-3
│       sensor:                            Airborne Visible / Infrared Imaging Sp...
│       instrument:                        AVIRIS-3
│       ...                                ...
│       ncei_template_version:             NCEI_NetCDF_Swath_Template_v2.0
│       title:                             AVIRIS-3 L1B Calibrated Radiance (flig...
│       processing_level:                  L1B
│       time_coverage_start:               2023-10-06T18:13:29Z
│       time_coverage_end:                 2023-10-06T18:14:05Z
│       product_version:                   1
├── Group: /geolocation_lookup_table
│       Dimensions:   (northing: 1760, easting: 1793)
│       Coordinates:
│         * northing  (northing) float64 14kB 3.546e+06 3.546e+06 ... 3.539e+06
│         * easting   (easting) float64 14kB 5.874e+05 5.874e+05 ... 5.946e+05 5.946e+05
│       Data variables:
│           sample    (northing, easting) float32 13MB ...
│           line      (northing, easting) float32 13MB ...
│       Attributes:
│           long_name:    Geolocation lookup table
│           description:  Orthocorrected product with a fixed pixel size projected in...
└── Group: /radiance
        Dimensions:     (wavelength: 284, lines: 1280, samples: 1234)
        Coordinates:
          * wavelength  (wavelength) float32 1kB 389.8 397.2 ... 2.486e+03 2.493e+03
        Dimensions without coordinates: lines, samples
        Data variables:
            radiance    (wavelength, lines, samples) float32 2GB ...
            fwhm        (wavelength) float32 1kB ...